### run environment python 3.10

In [196]:
!pip install numpy==1.26.4 tensorflow==2.15 deepchem==2.8.0 rdkit

In [197]:
import deepchem as dc

In [198]:
import numpy as np

In [199]:
tasks,dataset,transformers=dc.molnet.load_tox21()

In [200]:
print(tasks)

['NR-AR', 'NR-AR-LBD', 'NR-AhR', 'NR-Aromatase', 'NR-ER', 'NR-ER-LBD', 'NR-PPAR-gamma', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53']


| Task          | Meaning                   |
| ------------- | ------------------------- |
| NR-AR         | androgen receptor         |
| NR-AR-LBD     | androgen receptor binding |
| NR-AhR        | toxin receptor            |
| NR-Aromatase  | hormone conversion enzyme |
| NR-ER         | estrogen receptor         |
| NR-ER-LBD     | estrogen receptor binding |
| NR-PPAR-gamma | metabolism receptor       |
| SR-ARE        | oxidative stress          |
| SR-ATAD5      | DNA damage                |
| SR-HSE        | heat shock stress         |
| SR-MMP        | mitochondrial toxicity    |
| SR-p53        | tumor suppressor stress   |


In [201]:
len(tasks)

12

In [202]:
train_dataset,valid_dataset,test_dataset=dataset

In [203]:
train_dataset.X.shape

(6258, 1024)

In [204]:
train_dataset.y.shape

(6258, 12)

In [205]:
valid_dataset.X.shape

(782, 1024)

In [206]:
test_dataset.X.shape

(783, 1024)

In [207]:
np.shape(train_dataset.y)

(6258, 12)

In [208]:
np.shape(valid_dataset.y)

(782, 12)

In [209]:
np.shape(test_dataset.y)

(783, 12)

In [210]:
train_dataset.w.shape

(6258, 12)

In [211]:
np.count_nonzero(train_dataset.w)

63577

In [212]:
np.count_nonzero(train_dataset.w==0)

11519

In [213]:
transformers

In [214]:
from deepchem.models.fcnet import MultitaskClassifier


In [215]:
model=MultitaskClassifier(n_tasks=12,n_features=1024,layer_sizes=[1000])

In [216]:
model.fit(train_dataset,nb_epoch=10)

0.5137249946594238

In [217]:
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, np.mean)

In [218]:
print(metric.metric)

<function roc_auc_score at 0x7f1de23b6200>


In [219]:
train_scores = model.evaluate(train_dataset, [metric], transformers)
test_scores = model.evaluate(test_dataset, [metric], transformers)

In [220]:
print(train_scores)

{'mean-roc_auc_score': 0.9579139921560432}


In [221]:
print(test_scores)

{'mean-roc_auc_score': 0.6829093539568333}


In [222]:
import tensorflow as tf
import tensorflow.keras.layers as layers
    

In [223]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
y_train = tf.one_hot(y_train, 10).numpy()
y_test = tf.one_hot(y_test, 10).numpy()

In [224]:
train_dataset = dc.data.NumpyDataset(x_train, y_train)
test_dataset = dc.data.NumpyDataset(x_test, y_test)

In [225]:
features = tf.keras.Input(shape=(28, 28, 1))

In [226]:
conv2d_1 = layers.Conv2D(filters=32, kernel_size=5,
                         activation=tf.nn.relu)(features)
conv2d_2 = layers.Conv2D(filters=64, kernel_size=5,
                         activation=tf.nn.relu)(conv2d_1)

In [227]:
flatten = layers.Flatten()(conv2d_2)
dense1 = layers.Dense(units=1024, activation=tf.nn.relu)(flatten)
dense2 = layers.Dense(units=10, activation=None)(dense1)

In [228]:
output = layers.Activation(tf.math.softmax)(dense2)

In [229]:
keras_model = tf.keras.Model(inputs=features, outputs=[output, dense2])

In [231]:
from deepchem.models.keras_model import KerasModel

In [233]:
model = KerasModel(
    keras_model,
    loss=dc.models.losses.SoftmaxCrossEntropy(),
    output_types=['prediction', 'loss'],
    model_dir='mnist')

In [ ]:
isinstance(model, dc.models.Model)

True

In [ ]:
model.fit(train_dataset, nb_epoch=10)

In [ ]:
metric = dc.metrics.Metric(dc.metrics.accuracy_score)

In [ ]:
train_scores = model.evaluate(train_dataset, [metric])
test_scores = model.evaluate(test_dataset, [metric])

In [ ]:
print(test_scores)
print(train_scores)
